In [1]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import urllib.request
from datasets import load_dataset
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace

if torch.backends.mps.is_available():
    device = "mps"
elif torch.cuda.is_available():
    device = "cuda"
else:
    device = "cpu"
# torch.manual_seed(1337)
torch.cuda.manual_seed_all(1337) #gpu
print("device:", device)

/Users/shangyishen/Desktop/Study/Transformer/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


device: mps


In [2]:
dataset = load_dataset("wikitext", "wikitext-103-raw-v1")
texts = [x for x in dataset["train"]["text"] if x.strip()]

tokenizer = Tokenizer(BPE(unk_token="[UNK]"))
tokenizer.pre_tokenizer = Whitespace()

trainer = BpeTrainer(
    vocab_size=5000,
    special_tokens=["[UNK]"]
)

tokenizer.train_from_iterator(texts, trainer=trainer)


In [4]:
tokenizer.save("tokenizer.json")

In [5]:
all_ids = []
for i, t in enumerate(texts):
    if not t.strip():
        continue
    all_ids.extend(tokenizer.encode(t).ids)
    if i % 5000 == 0:
        print(f"encoded {i} lines")

data = torch.tensor(all_ids, dtype=torch.long)
vocab_size = tokenizer.get_vocab_size()

def encode(s):
    return tokenizer.encode(s).ids

def decode(t):
    return tokenizer.decode([int(i) for i in t])

print("vocab_size:", vocab_size, "data_len:", len(data))

# 保存，明天直接加载
tokenizer.save("tokenizer.json")
torch.save(data, "dataset.pt")

block_size = 128
batch_size = 16

def get_batch():
    ix = torch.randint(0, len(data) - block_size - 1, (batch_size,))
    x = torch.stack([data[i:i + block_size] for i in ix])
    y = torch.stack([data[i + 1:i + block_size + 1] for i in ix])
    return x.to(device), y.to(device)

xb, yb = get_batch()
print("xb:", xb.shape, "yb:", yb.shape)

sample = "To be or not to be"
sample_ids = encode(sample)
print("encoded:", sample_ids)
print("decoded:", decode(sample_ids))

encoded 0 lines
encoded 5000 lines
encoded 10000 lines
encoded 15000 lines
encoded 20000 lines
encoded 25000 lines
encoded 30000 lines
encoded 35000 lines
encoded 40000 lines
encoded 45000 lines
encoded 50000 lines
encoded 55000 lines
encoded 60000 lines
encoded 65000 lines
encoded 70000 lines
encoded 75000 lines
encoded 80000 lines
encoded 85000 lines
encoded 90000 lines
encoded 95000 lines
encoded 100000 lines
encoded 105000 lines
encoded 110000 lines
encoded 115000 lines
encoded 120000 lines
encoded 125000 lines
encoded 130000 lines
encoded 135000 lines
encoded 140000 lines
encoded 145000 lines
encoded 150000 lines
encoded 155000 lines
encoded 160000 lines
encoded 165000 lines
encoded 170000 lines
encoded 175000 lines
encoded 180000 lines
encoded 185000 lines
encoded 190000 lines
encoded 195000 lines
encoded 200000 lines
encoded 205000 lines
encoded 210000 lines
encoded 215000 lines
encoded 220000 lines
encoded 225000 lines
encoded 230000 lines
encoded 235000 lines
encoded 240000 li

In [6]:
def make_causal_mask(T: int, device=None):
    mask=torch.tril(torch.ones(T, T, device=device))
    return mask

T = 5
mask = make_causal_mask(T, device=device)
print(mask)

tensor([[1., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0.],
        [1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1.]], device='mps:0')


In [7]:
class SingleHeadCausalSelfAttention(nn.Module):
    def __init__(self, n_embd: int, head_dim: int, block_size: int): #block_size:模型允许的最大长度
        super().__init__()
        self.head_dim=head_dim
        self.q= nn.Linear(n_embd,head_dim)
        self.k= nn.Linear(n_embd,head_dim)
        self.v=nn.Linear(n_embd,head_dim)
        self.register_buffer("mask", torch.tril(torch.ones(block_size, block_size))) #保留下三角，防止预测时看到未来的token

    def forward(self, x):
        # x: (B,T,C)
        # TODO: implement causal attention, return (B,T,head_dim)
        B,T,C = x.shape
        q = self.q(x) # (B, T, head_dim)
        k = self.k(x) # (B, T, head_dim)
        v = self.v(x) # (B, T, head_dim)
        att = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)   #(T,head_dim)@(head_dim,T)->(T,T)
        att = att.masked_fill(self.mask[:T, :T] == 0, float("-inf")) #注释在下面
        weight= F.softmax(att, dim=-1) #这里-1不是自动找维度，attn.shape=[B,T(query token),T(key_token)]所以输出维度应该与key_token维度一致
        out_put = weight @ v
        return out_put

# quick shape test
B, T, C = 4, 8, 32
x = torch.randn(B, T, C)
attn = SingleHeadCausalSelfAttention(n_embd=C, head_dim=8, block_size=block_size)
out = attn(x)
print("out shape:", out.shape)

out shape: torch.Size([4, 8, 8])


In [12]:
class MultiHeadCausalSelfAttention(nn.Module):
    def __init__(self, n_embd: int, n_head: int, block_size: int, dropout: float = 0.0):
        super().__init__()
        assert n_embd % n_head == 0
        self.n_head = n_head
        self.head_dim = n_embd // n_head

        self.qkv = nn.Linear(n_embd, 3 * n_embd, bias=False)
        self.proj = nn.Linear(n_embd, n_embd, bias=False)
        self.gate = nn.Linear(n_embd, n_embd, bias=False)
        self.register_buffer("mask", torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        qkv = self.qkv(x)
        q, k, v = qkv.split(C, dim=-1)

        q = q.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, self.head_dim).transpose(1, 2)

        attn_qk = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        attn_qk = attn_qk.masked_fill(self.mask[:T, :T] == 0, float("-inf"))
        attn_weight = F.softmax(attn_qk, dim=-1)
        attn_weight = self.dropout(attn_weight)

        attn = attn_weight @ v
        attn = attn.transpose(1, 2).contiguous().view(B, T, C)

        gate = torch.sigmoid(self.gate(attn))
        attn = self.proj(attn)
        attn = attn * gate
        attn = self.dropout(attn)
        return attn

In [13]:
class GatedFeedForward(nn.Module):
    def __init__(self, n_embd: int, dropout: float = 0.0):
        super().__init__()
        hidden = 4 * n_embd
        self.fc = nn.Linear(n_embd, hidden)
        self.gate = nn.Linear(n_embd, hidden)
        self.proj = nn.Linear(hidden, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        h = F.gelu(self.fc(x))                 # 内容分支
        g = torch.sigmoid(self.gate(x))       # 门控分支
        x = h * g
        x = self.proj(x)
        x = self.dropout(x)
        return x


# shape test
x = torch.randn(3, 7, 32)
ff = GatedFeedForward(32)
y = ff(x)
print("y shape:", y.shape)

y shape: torch.Size([3, 7, 32])


In [14]:
class TransformerBlock(nn.Module):
    def __init__(self, n_embd: int, n_head: int, block_size: int, dropout: float = 0.0):
        super().__init__()
        self.ln1 = nn.LayerNorm(n_embd)
        self.mha = MultiHeadCausalSelfAttention(
            n_embd=n_embd,
            n_head=n_head,
            block_size=block_size,
            dropout=dropout,
        )
        self.ln2 = nn.LayerNorm(n_embd)
        self.mlp = GatedFeedForward(n_embd, dropout=dropout)

    def forward(self, x):
        x = x + self.mha(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x



# shape test
x = torch.randn(2, 16, 64)
blk = TransformerBlock(n_embd=64, n_head=4, block_size=block_size)
y = blk(x)
print("y shape:", y.shape)

y shape: torch.Size([2, 16, 64])


In [15]:
class TinyGPT(nn.Module):
    def __init__(self, vocab_size: int, block_size: int, n_embd: int = 128, n_head: int = 4, n_layer: int = 4, dropout: float = 0.1):
        super().__init__()
        self.block_size = block_size
        self.tok_emb = nn.Embedding(vocab_size, n_embd)
        self.pos_emb = nn.Embedding(block_size, n_embd)
        self.blocks = nn.ModuleList([TransformerBlock(n_embd, n_head, block_size, dropout) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.head = nn.Linear(n_embd, vocab_size)


    def forward(self, idx, targets=None):
        B, T = idx.shape
        assert T <= self.block_size
        pos = torch.arange(T, device=idx.device)
        x = self.tok_emb(idx) + self.pos_emb(pos)[None, :, :]
        for blk in self.blocks:
            x = blk(x)
        x = self.ln_f(x)
        logits = self.head(x)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss

# quick forward test
model = TinyGPT(vocab_size=vocab_size, block_size=block_size).to(device)
xb, yb = get_batch()
logits, loss = model(xb, yb)
print("logits:", logits.shape, "loss:", float(loss))

logits: torch.Size([16, 128, 5000]) loss: 8.674580574035645


In [17]:
@torch.no_grad()
def generate(model, idx, max_new_tokens=100, temperature=1.0, top_k=None):
    model.eval()
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -model.block_size:]
        logits, _ = model(idx_cond)
        next_logits = logits[:, -1, :] / temperature

        if top_k is not None:
            v, _ = torch.topk(next_logits, min(top_k, next_logits.size(-1)))
            next_logits[next_logits < v[:, [-1]]] = -float("inf")

        probs = F.softmax(next_logits, dim=-1)
        next_idx = torch.multinomial(probs, num_samples=1)
        idx = torch.cat([idx, next_idx], dim=1)
    return idx
model = TinyGPT(vocab_size=vocab_size, block_size=block_size, n_embd=128, n_head=4, n_layer=4).to(device)
opt = torch.optim.AdamW(model.parameters(), lr=3e-3)

for step in range(10000):
    xb, yb = get_batch()
    logits, loss = model(xb, yb)
    opt.zero_grad(set_to_none=True)
    loss.backward()
    opt.step()
    if step % 100 == 0:
        print("step", step, "loss", float(loss))

# try generating
start = torch.zeros((1, 1), dtype=torch.long, device=device)  # start token id 0 (arbitrary)
out = generate(model, start, max_new_tokens=120, temperature=0.9)
print(decode(out[0].tolist()))

step 0 loss 8.710403442382812
step 100 loss 3.4508018493652344
step 200 loss 3.3382298946380615
step 300 loss 3.3976690769195557
step 400 loss 3.3236680030822754
step 500 loss 3.3536057472229004
step 600 loss 3.3064522743225098
step 700 loss 3.2343828678131104
step 800 loss 3.1549391746520996
step 900 loss 3.0914480686187744
step 1000 loss 3.0746495723724365
step 1100 loss 3.08302640914917
step 1200 loss 3.0224997997283936
step 1300 loss 2.955899953842163
step 1400 loss 2.9221274852752686
step 1500 loss 2.9281883239746094
step 1600 loss 2.9197440147399902
step 1700 loss 2.829080820083618
step 1800 loss 2.790048599243164
step 1900 loss 2.852546215057373
step 2000 loss 2.8060355186462402
step 2100 loss 2.9165871143341064
step 2200 loss 2.702342987060547
step 2300 loss 2.950599193572998
step 2400 loss 2.739966869354248
step 2500 loss 2.7413535118103027
step 2600 loss 2.6853115558624268
step 2700 loss 2.750807046890259
step 2800 loss 2.706347942352295
step 2900 loss 2.5955233573913574
step